## Comparing .json files

In [22]:
import json

def verify_alignment(file1, file2):
    A = json.load(open(file1))
    B = json.load(open(file2))
    
    for split in ["train", "val", "test"]:
        imgs_a = {s["image"] for s in A[split]}
        imgs_b = {s["image"] for s in B[split]}
        print(f"Len A: {len(imgs_a)}")
        print(f"Len B: {len(imgs_b)}")
        
        if imgs_a == imgs_b:
            print(f"✅ {split}: {len(imgs_a)} images match")
        else:
            print(f"❌ {split}: Mismatch!")
            print(f"   Only in A: {len(imgs_a - imgs_b)}")
            print(f"   Only in B: {len(imgs_b - imgs_a)}")

# verify_alignment(
#     r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified_10k.json",
#     r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified_meanrgb_10k.json"
# )

In [20]:
verify_alignment(
    r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\Professions_125k_ISCO_Aligned_1k_Subset\splits_gender_face_stratified.json",
    r"F:\ImageRetrieval\SpuriousFeatureImages\Professions_125k_ISCO_Aligned_1k_Subset\_SPLITS\Shuffling&Colour\mean_rgb\splits_gender_face_stratified_meanrgb_normalized.json",
)

Len A: 130193
Len B: 130193
✅ train: 130193 images match
Len A: 27897
Len B: 27897
✅ val: 27897 images match
Len A: 27901
Len B: 27901
✅ test: 27901 images match


In [22]:
verify_alignment(
    r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\StableDiffusion\splits_gender_face_stratified.json",
    r"F:\ImageRetrieval\SpuriousFeatureImages\StableDiffusionImages\_SPLITS\Shuffling&Colour\mean_rgb\splits_gender_face_stratified_meanrgb_normalized.json",
)

Len A: 139999
Len B: 139999
✅ train: 139999 images match
Len A: 29999
Len B: 29999
✅ val: 29999 images match
Len A: 30002
Len B: 30002
✅ test: 30002 images match


In [23]:
verify_alignment(
    r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
    r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified_meanrgb.json",
)

Len A: 270192
Len B: 270192
✅ train: 270192 images match
Len A: 57896
Len B: 57896
✅ val: 57896 images match
Len A: 57903
Len B: 57903
✅ test: 57903 images match


In [25]:
verify_alignment(
    r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified_10k.json",
    r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified_meanrgb_10k.json",
)

Len A: 9999
Len B: 9999
✅ train: 9999 images match
Len A: 9999
Len B: 9999
✅ val: 9999 images match
Len A: 9999
Len B: 9999
✅ test: 9999 images match


In [27]:
import json
from pathlib import Path

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
INPUT_JSON  = Path(r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified_meanrgb_10k.json")
OUTPUT_JSON = Path(r"output_same_features.json")

# --------------------------------------------------
# LOAD
# --------------------------------------------------
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

train = data["train"]

if len(train) == 0:
    raise ValueError("Train split is empty")

# --------------------------------------------------
# REFERENCE FEATURES (take from first sample)
# --------------------------------------------------
shared_features = train[0]["features"]

# Optional: sanity check
assert isinstance(shared_features, list), "Features must be a list"

# --------------------------------------------------
# OVERWRITE ALL TRAIN FEATURES
# --------------------------------------------------
for sample in train:
    sample["features"] = shared_features

# --------------------------------------------------
# SAVE
# --------------------------------------------------
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print(f"Saved modified dataset to {OUTPUT_JSON}")


Saved modified dataset to output_same_features.json


# Updating Dataset Classifciation Image Mappings for Other Image Variants

In [8]:
from pathlib import Path

def replace_in_file(
    file_path: str,
    old: str,
    new: str,
    encoding: str = "utf-8"
):
    path = Path(file_path)

    text = path.read_text(encoding=encoding)
    text = text.replace(old, new)
    path.write_text(text, encoding=encoding)

    print(f"Replaced all occurrences of '{old}' with '{new}' in {path}")

In [20]:
import json
from pathlib import Path
import re

# Matches: _depth, _edges, _canny, _seg, _mask, etc. at end of name
SUFFIX_RE = re.compile(r"_(depth|edges|edge|canny|seg|mask|mean|high|low|ps16|ps8|ps4|ps2|vae|pixel|)$", re.IGNORECASE)

def normalize_image_name(filename: str) -> str:
    """
    Normalize an image filename by:
    - removing extension
    - removing known variant suffixes
    """
    stem = Path(filename).stem
    stem = SUFFIX_RE.sub("", stem)
    return stem


def image_key(path: str):
    """
    Canonical image identity:
      (parent_dir, normalized_image_stem)
    """
    p = Path(path)
    return (p.parent.name, normalize_image_name(p.name))


def verify_alignment(file1, file2):
    A = json.load(open(file1))
    B = json.load(open(file2))
    
    for split in ["train", "val", "test"]:
        imgs_a = {image_key(s["image"]) for s in A[split]}
        imgs_b = {image_key(s["image"]) for s in B[split]}

        print(f"\n[{split}]")
        print(f"Len A: {len(imgs_a)}")
        print(f"Len B: {len(imgs_b)}")

        if imgs_a == imgs_b:
            print(f"✅ {split}: {len(imgs_a)} images match (normalized)")
        else:
            print(f"❌ {split}: Mismatch!")
            only_a = imgs_a - imgs_b
            only_b = imgs_b - imgs_a

            print(f"   Only in A: {len(only_a)}")
            print(f"   Only in B: {len(only_b)}")
            print("   Example only-in-A:", list(only_a)[:3])
            print("   Example only-in-B:", list(only_b)[:3])


### Depth

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_depth.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\Depth",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Depth",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_depth.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_depth.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Depth' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Depth\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_depth.json


In [46]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_depth.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### EdgeDetection

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_edge.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\EdgeDetection\\edges_canny",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\EdgeDetection\\edges_canny",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_edges.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_edges.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\EdgeDetection\\edges_canny' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\EdgeDetection\\edges_canny\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_edge.json


In [43]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_edge.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### HighFilter

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_highFilter.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\High&LowPassFilter\\ideal\\radius_40\\high_pass",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\High&LowPassFilter\\ideal\\radius_40\\high_pass",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_high.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_high.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\High&LowPassFilter\\ideal\\radius_40\\high_pass' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\High&LowPassFilter\\ideal\\radius_40\\high_pass\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_highFilter.json


In [49]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_highFilter.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### LowFilter

In [25]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_lowFilter.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\High&LowPassFilter\\ideal\\radius_40\\low_pass",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\High&LowPassFilter\\ideal\\radius_40\\low_pass",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_low.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_low.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\High&LowPassFilter\\ideal\\radius_40\\low_pass' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_lowFilter.json
Replaced all occurrences of 'E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\High&LowPassFilter\\ideal\\radius_40\\low_pass' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_lowFilter.json
Replaced all occurrences of '.png' with '_low.png' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_lowFilter.json
Replaced all occurrences of '.jpg' with '_low.png' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_lowFilter.json


In [26]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_lowFilter.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### MeanRGB

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_meanRGB.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\Shuffling&Colour\\mean_rgb",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\mean_rgb",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_mean.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_mean.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\mean_rgb' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\mean_rgb\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_meanRGB.json


In [56]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_meanRGB.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### objectDetection (White Background)

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_objectDetection.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\ObjectDetection\\white_background",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\ObjectDetection\\white_background",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r".png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r".png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\ObjectDetection\\white_background' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\ObjectDetection\\white_background\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_objectDetection.json


In [58]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_objectDetection.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### PatchShufflePS2

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS2.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\Shuffling&Colour\\patch_shuffle_ps2",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps2",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_ps2.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_ps2.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps2' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps2\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS2.json


In [3]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS2.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### PatchShufflePS4

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS4.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\Shuffling&Colour\\patch_shuffle_ps4",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps4",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_ps4.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_ps4.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps4' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps4\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS4.json


In [4]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS4.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### PatchShufflePS8

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS8.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\Shuffling&Colour\\patch_shuffle_ps8",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps8",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_ps8.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_ps8.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps8' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps8\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS8.json


In [5]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS8.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### PatchShufflePS16

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS16.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\Shuffling&Colour\\patch_shuffle_ps16",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps16",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_ps16.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_ps16.png",
)


Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps16' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\patch_shuffle_ps16\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS16.json


In [11]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_patchShufflePS16.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### PixelShuffle

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_pixelShuffle.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\Shuffling&Colour\\pixel_shuffle",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\pixel_shuffle",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_pixel.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_pixel.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\pixel_shuffle' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\Shuffling&Colour\\pixel_shuffle\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_pixelShuffle.json


In [21]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_pixelShuffle.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### SemanticSegmentation

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_semanticSegmentation.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\SemanticSegmentation",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\SemanticSegmentation",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_seg.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_seg.png",
)

Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\SemanticSegmentation' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\SemanticSegmentation\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_semanticSegmentation.json


In [15]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_semanticSegmentation.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


### VAE

In [ ]:
fp = r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_vae.json"

replace_in_file(
    file_path=fp,
    old=r"F:\\ImageRetrieval\\Professions_125k_ISCO_Aligned_1k_Subset",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\Professions_125k_ISCO_Aligned_1k_Subset\\VAE",
)

replace_in_file(
    file_path=fp,
    old=r"E:\\ImageRetrieval\\StableDiffusionGeneratedImages\\valid",
    new=r"F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\VAE",
)

replace_in_file(
    file_path=fp,
    old=r".png",
    new=r"_vae.png",
)

replace_in_file(
    file_path=fp,
    old=r".jpg",
    new=r"_vae.png",
)


Replaced all occurrences of 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\VAE' with 'F:\\ImageRetrieval\\SpuriousFeatureImages\\StableDiffusionImages\\VAE\\' in UniversalSplits\DatasetClassification\splits_face_combined_stratified_vae.json


In [19]:
verify_alignment(
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified_vae.json",
    r"UniversalSplits\DatasetClassification\splits_face_combined_stratified.json",
)


[train]
Len A: 270192
Len B: 270192
✅ train: 270192 images match (normalized)

[val]
Len A: 57896
Len B: 57896
✅ val: 57896 images match (normalized)

[test]
Len A: 57903
Len B: 57903
✅ test: 57903 images match (normalized)


## COCO TEST

In [10]:
import json
from pathlib import Path

# ==========================
# CONFIG
# ==========================
BASE_SPLITS_JSON = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\splits_face_combined_stratified_edge.json"
OUTPUT_SPLITS_JSON = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassificationCoco\coco_splits_face_combined_stratified_edge.json"

COCO_ROOT = Path(r"F:\ImageRetrieval\SpuriousFeatureImages\Coco\EdgeDetection\edges_canny")
COCO_SPLITS = {
    "train": COCO_ROOT / "train2017",
    "val":   COCO_ROOT / "val2017",
    "test":  COCO_ROOT / "test2017",  # remove if you don't want test
}

COCO_LABEL = 2
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

# ==========================
# LOAD EXISTING SPLITS
# ==========================
with open(BASE_SPLITS_JSON, "r", encoding="utf-8") as f:
    splits = json.load(f)

# Ensure all splits exist
for split in ["train", "val", "test"]:
    splits.setdefault(split, [])

# ==========================
# ADD COCO IMAGES
# ==========================
for split_name, split_dir in COCO_SPLITS.items():
    if not split_dir.exists():
        print(f"⚠ Skipping missing split: {split_dir}")
        continue

    coco_images = [
        p for p in split_dir.rglob("*")
        if p.suffix.lower() in IMAGE_EXTS
    ]

    print(f"Adding {len(coco_images):,} COCO images to '{split_name}'")

    for img_path in coco_images:
        splits[split_name].append({
            "image": str(img_path.resolve()),
            "label": COCO_LABEL
        })

# ==========================
# SAVE NEW JSON
# ==========================
with open(OUTPUT_SPLITS_JSON, "w", encoding="utf-8") as f:
    json.dump(splits, f, indent=2)

print(f"\n✅ Saved merged splits to:\n{OUTPUT_SPLITS_JSON}")


Adding 118,287 COCO images to 'train'
Adding 5,000 COCO images to 'val'
Adding 40,670 COCO images to 'test'

✅ Saved merged splits to:
C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassificationCoco\coco_splits_face_combined_stratified_edge.json


In [6]:
import json
from collections import Counter

# Path to your JSON file
file_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\added_coco_splits_face_combined_stratified_depth.json'

def calculate_percentage_splits(path):
    with open(path, 'r') as f:
        data = json.load(f)

    # Define the specific splits to analyze
    target_splits = ['train', 'val', 'test']
    
    # Calculate total samples across valid splits only
    total_samples = sum(len(data[s]) for s in target_splits if s in data)
    
    print(f"Total Samples in Dataset: {total_samples}\n")
    print(f"{'Split':<12} | {'Count':<10} | {'% Split':<10} | {'L0 %':<8} | {'L1 %':<8} | {'L2 %':<8}")
    print("-" * 75)

    for split_name in target_splits:
        if split_name not in data:
            continue
            
        entries = data[split_name]
        count = len(entries)
        split_percent = (count / total_samples) * 100
        
        # Count labels (ensure we are looking at dictionaries)
        label_counts = Counter(item.get('label') for item in entries if isinstance(item, dict))
        
        # Calculate percentages for labels 0, 1, and 2
        l0_pct = (label_counts.get(0, 0) / count) * 100 if count > 0 else 0
        l1_pct = (label_counts.get(1, 0) / count) * 100 if count > 0 else 0
        l2_pct = (label_counts.get(2, 0) / count) * 100 if count > 0 else 0

        print(f"{split_name:<12} | {count:<10} | {split_percent:>8.2f}% | {l0_pct:>6.2f}% | {l1_pct:>6.2f}% | {l2_pct:>6.2f}%")

if __name__ == "__main__":
    calculate_percentage_splits(file_path)

Total Samples in Dataset: 549948

Split        | Count      | % Split    | L0 %     | L1 %     | L2 %    
---------------------------------------------------------------------------
train        | 388479     |    70.64% |  33.51% |  36.04% |  30.45%
val          | 62896      |    11.44% |  44.35% |  47.70% |   7.95%
test         | 98573      |    17.92% |  28.30% |  30.44% |  41.26%


In [7]:
import json
from collections import Counter

file_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\7_DatasetPreparation\UniversalSplits\DatasetClassification\added_coco_splits_face_combined_stratified_depth.json'

def calculate_filtered_percentage_splits(path):
    with open(path, 'r') as f:
        data = json.load(f)

    target_splits = ['train', 'val', 'test']
    
    # Filter entries to only include Label 0 and Label 1
    split_data_filtered = {}
    total_samples_filtered = 0
    
    for split_name in target_splits:
        if split_name not in data:
            continue
            
        entries = data[split_name]
        # Only keep items where label is 0 or 1
        filtered = [item for item in entries if isinstance(item, dict) and item.get('label') in [0, 1]]
        split_data_filtered[split_name] = filtered
        total_samples_filtered += len(filtered)
    
    print(f"Total Samples (Labels 0 & 1 only): {total_samples_filtered}\n")
    print(f"{'Split':<12} | {'Count (0+1)':<12} | {'% Split':<10} | {'Label 0 %':<12} | {'Label 1 %':<12}")
    print("-" * 80)

    for split_name in target_splits:
        if split_name not in split_data_filtered:
            continue
            
        entries = split_data_filtered[split_name]
        count = len(entries)
        split_percent = (count / total_samples_filtered) * 100 if total_samples_filtered > 0 else 0
        
        label_counts = Counter(item.get('label') for item in entries)
        
        # Percentages relative to the new filtered split total
        l0_pct = (label_counts.get(0, 0) / count) * 100 if count > 0 else 0
        l1_pct = (label_counts.get(1, 0) / count) * 100 if count > 0 else 0

        print(f"{split_name:<12} | {count:<12} | {split_percent:>8.2f}% | {l0_pct:>10.2f}% | {l1_pct:>10.2f}%")

if __name__ == "__main__":
    calculate_filtered_percentage_splits(file_path)

Total Samples (Labels 0 & 1 only): 385991

Split        | Count (0+1)  | % Split    | Label 0 %    | Label 1 %   
--------------------------------------------------------------------------------
train        | 270192       |    70.00% |      48.19% |      51.81%
val          | 57896        |    15.00% |      48.18% |      51.82%
test         | 57903        |    15.00% |      48.19% |      51.81%
